In [1]:
import os
import re
import zipfile
import yaml

input_dir = r"C:\Users\User\Documents\Skillmap\japanese\tests\old"      # Path to folder containing your YAML files
output_dir = r"C:\Users\User\Documents\Skillmap\japanese\tests"  # Path to save fixed YAML files

os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    if filename.endswith(('.yaml', '.yml')):
        filepath = os.path.join(input_dir, filename)
        
        with open(filepath, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f)

        # Handle list of questions or single item
        items = data if isinstance(data, list) else [data]

        for item in items:
            if isinstance(item, dict) and "options" in item and "correct_answer" in item:
                ans = str(item["correct_answer"]).strip()
                # Find matching option that starts with "X:"
                for opt in item["options"]:
                    if opt.startswith(f"{ans}:"):
                        item["correct_answer"] = opt
                        break

        # Save fixed YAML
        out_filepath = os.path.join(output_dir, filename)
        with open(out_filepath, 'w', encoding='utf-8') as f:
            yaml.dump(data, f, allow_unicode=True, sort_keys=False)


## Fix Explanation in Revision

In [6]:
import json
import yaml
import os

def normalize_text(text):
    """Normalize text for reliable matching by stripping whitespaces and normalizing newlines."""
    if not text:
        return ""
    # Standardize windows/linux line breaks and strip extra spaces around lines
    lines = [line.strip() for line in str(text).replace('\r\n', '\n').split('\n') if line.strip()]
    return "\n".join(lines)

def fix_json_with_yaml(json_path, yaml_dir, output_path):
    # 1. Load the JSON file
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # 2. Read and build an explanation mapping from YAML file(s)
    explanation_map = {}
    
    if os.path.isdir(yaml_dir):
        for filename in os.listdir(yaml_dir):
            if filename.endswith(('.yaml', '.yml')):
                with open(os.path.join(yaml_dir, filename), 'r', encoding='utf-8') as yf:
                    yaml_data = yaml.safe_load(yf)
                    
                    # Support both a top-level list or a dictionary containing a 'questions' list
                    questions_list = []
                    if isinstance(yaml_data, list):
                        questions_list = yaml_data
                    elif isinstance(yaml_data, dict):
                        questions_list = yaml_data.get('questions', [])
                        
                    for entry in questions_list:
                        if isinstance(entry, dict):
                            # Normalize the entire block of question + options text from YAML
                            y_question_block = normalize_text(entry.get('question'))
                            explanation = entry.get('explanation')
                            
                            if y_question_block and explanation:
                                explanation_map[y_question_block] = explanation

    # 3. Traverse JSON 'records' array and inject explanations
    if isinstance(data, dict) and "records" in data:
        for record in data["records"]:
            # Normalize the JSON question_name block the same way
            json_question_block = normalize_text(record.get('question_name', ''))
            
            # Match directly using the full normalized text block (Question + Options)
            if json_question_block in explanation_map:
                record['explanation'] = explanation_map[json_question_block]
            else:
                # Fallback: Match by just the first line (the core sentence) if blocks differ slightly
                core_sentence = json_question_block.split('\n')[0] if json_question_block else ""
                matched = False
                
                for y_block, expl in explanation_map.items():
                    y_core = y_block.split('\n')[0] if y_block else ""
                    if core_sentence and core_sentence == y_core:
                        record['explanation'] = expl
                        matched = True
                        break
                        
                if not matched:
                    print(f"Warning: No match found for record ID {record.get('id', 'unknown')}")

    # 4. Save the fixed JSON file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
        
    print(f"Successfully processed JSON records and saved to {output_path}")

# --- Example Usage ---
if __name__ == "__main__":
    fix_json_with_yaml(
        json_path=r"C:\Users\User\Documents\Skillmap\japanese\revision\revision-data-20260802-014200.json",       # Path to your input JSON
        yaml_dir=r"C:\Users\User\Documents\Skillmap\japanese\tests",      # Folder containing your YAML explanation files
        output_path=r"C:\Users\User\Documents\Skillmap\japanese\revision\revision-data-20260802-014200-fixed.json"# Path where the updated JSON will be saved
    )

Successfully processed JSON records and saved to C:\Users\User\Documents\Skillmap\japanese\revision\revision-data-20260802-014200-fixed.json


## Split 7 YAMLs into files

In [2]:
import yaml
import os
import re

def split_big_yaml(input_filepath, output_dir="split_yamls", yaml_filname_prefix="jp-n3-w2", overwrite=False):
    """
    Reads a large multi-quiz YAML file separated by test headers and splits 
    it into individual daily YAML files named jp-n3-w#-d{day}-test.yaml.
    """
    if not os.path.exists(input_filepath):
        print(f"Error: File not found -> {input_filepath}")
        return

    os.makedirs(output_dir, exist_ok=True)

    with open(input_filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # Split the content by the test header comments
    sections = content.split('# ==========================================')
    
    day_counter = 1

    for section in sections:
        section_stripped = section.strip()
        if not section_stripped:
            continue

        # Look for the quiz_title line or extract day number from title text
        quiz_title_match = re.search(r'quiz_title:\s*["\'](.*?)["\']', section_stripped)
        
        if quiz_title_match:
            title_text = quiz_title_match.group(1)
            
            # Try to extract the day number from the title string (e.g., "1日目のテスト" -> 1)
            day_num_match = re.search(r'(\d+)日目', title_text)
            if day_num_match:
                current_day = int(day_num_match.group(1))
            else:
                current_day = day_counter
                
            # Clean up leading/trailing dashes, comments, or stray YAML document separators
            clean_lines = []
            for line in section_stripped.split('\n'):
                stripped_line = line.strip()
                # Skip comment lines and standalone YAML document separators (---)
                if stripped_line.startswith('#') or stripped_line == '---':
                    continue
                clean_lines.append(line)
                
            clean_content = '\n'.join(clean_lines).strip()
            
            # Load YAML safely using safe_load_all or handling single documents cleanly
            try:
                # safe_load_all handles edge cases where hidden document markers might slip through
                docs = list(yaml.safe_load_all(clean_content))
                parsed_data = docs[0] if docs else None
                
                if parsed_data and isinstance(parsed_data, dict) and 'questions' in parsed_data:
                    output_filename = f"{yaml_filname_prefix}-d{current_day}-test.yaml"
                    output_path = os.path.join(output_dir, output_filename)
                    
                    if os.path.exists(output_path) and not overwrite:
                        print(f"Skipped (already exists): {output_path}")
                        day_counter += 1
                        continue

                    with open(output_path, 'w', encoding='utf-8') as out_f:
                        yaml.dump(parsed_data, out_f, allow_unicode=True, sort_keys=False, indent=2)
                        
                    print(f"Created: {output_path}")
                    day_counter += 1
            except yaml.YAMLError as ye:
                print(f"YAML parsing error in section for Day {current_day}: {ye}")

    print(f"\nSuccessfully processed file into folder: '{output_dir}/'")

# --- Example Usage ---
if __name__ == "__main__":
    split_big_yaml(
        input_filepath=r"C:\Users\User\Documents\Skillmap\japanese\tests\gemini-code-1785683733529.yaml",  
        output_dir=r"C:\Users\User\Documents\Skillmap\japanese\tests",         
        yaml_filname_prefix="jp-n3-w2",
        overwrite=False
    )

Created: C:\Users\User\Documents\Skillmap\japanese\tests\jp-n3-w2-d1-test.yaml
Created: C:\Users\User\Documents\Skillmap\japanese\tests\jp-n3-w2-d2-test.yaml
Created: C:\Users\User\Documents\Skillmap\japanese\tests\jp-n3-w2-d3-test.yaml
Created: C:\Users\User\Documents\Skillmap\japanese\tests\jp-n3-w2-d4-test.yaml
Created: C:\Users\User\Documents\Skillmap\japanese\tests\jp-n3-w2-d5-test.yaml
Created: C:\Users\User\Documents\Skillmap\japanese\tests\jp-n3-w2-d6-test.yaml
Skipped (already exists): C:\Users\User\Documents\Skillmap\japanese\tests\jp-n3-w2-d7-test.yaml

Successfully processed file into folder: 'C:\Users\User\Documents\Skillmap\japanese\tests/'
